# PV forecast error generalization notebook - aligned with the wind workflow

## Strategy Overview
1. Generalize errors only for timestamps with generation output (zero-output points are filtered out)
2. Follow the wind KDE + Copula workflow exactly
3. The generated error samples cover daytime periods only (48 points)
4. In downstream use, set nighttime periods directly to zero

In [ ]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde, norm, multivariate_normal
from scipy.special import ndtr
from scipy import interpolate

## 1. Data loading and preprocessing

In [ ]:
print("=== 1. Data loading and preprocessing ===")

# Load PV data
df = pd.read_csv('data/raw/ods032.csv', sep=';')
print(f"Original number of rows: {len(df)}")

# Convert timestamps
df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True)

# Extract the required columns
actual_values = df['Measured & Upscaled'].values
forecast_values = df['Most recent forecast'].values

In [ ]:
# Filter nighttime data (PV-specific step)
# Keep only rows where either the actual value or the forecast is nonzero (daytime generation periods)
mask = (actual_values != 0) | (forecast_values != 0)
actual_daytime = actual_values[mask]
forecast_daytime = forecast_values[mask]

print(f"Rows after nighttime filtering: {len(actual_daytime)}")

## 2. Forecast error calculation

In [ ]:
print("\n=== 2. Forecast error calculation ===")

# Compute forecast error (actual value - forecast value)
forecast_error = actual_daytime - forecast_daytime

# Normalize forecast error by the standard deviation of actual values
forecast_error_normalized = forecast_error / np.std(actual_daytime)

print(f"Error statistics:")
print(f"  Mean: {np.mean(forecast_error_normalized):.4f}")
print(f"  Std: {np.std(forecast_error_normalized):.4f}")
print(f"  Min: {np.min(forecast_error_normalized):.4f}")
print(f"  Max: {np.max(forecast_error_normalized):.4f}")

## 3. Reshape into a time-series structure (strictly following the wind method)

In [ ]:
print("\n=== 3. Reshape into a time-series structure ===")

# MethodFollow the wind workflow and use a fixed time window
# Use 48 points as one period here, consistent with the wind workflow
# Note: these 48 points all correspond to active generation periods and exclude nighttime
num_time_points = 48  # fixed window size, consistent with the wind workflow
num_days = len(forecast_error_normalized) // num_time_points

# Trim to complete periods
forecast_error_normalized = forecast_error_normalized[:num_days * num_time_points]

#  (Number of periods, num_time_points)
forecast_error_matrix = forecast_error_normalized.reshape(num_days, num_time_points)

print(f"Error matrix shape: {forecast_error_matrix.shape}")
print(f"  Number of periods: {num_days}")
print(f"  Time points per period: {num_time_points}")
print(f"  Note: all 48 points correspond to active generation periods, with nighttime zeros removed")

## 4. Temporal correlation analysis

In [ ]:
print("\n=== 4. Temporal correlation analysis ===")

# Temporal correlation analysis
temp_corr = np.corrcoef(forecast_error_matrix, rowvar=False)
print(f"Temporal-correlation matrix shape: {temp_corr.shape}")

In [ ]:
# Visualize the temporal-correlation heatmap
plt.figure(figsize=(10, 10))
mask_upper = np.triu(np.ones_like(temp_corr, dtype=bool))
sns.heatmap(temp_corr, annot=False, fmt=".2f", mask=mask_upper, cmap='coolwarm')
plt.title('Solar Forecasting Error Temporal Correlation')
plt.tight_layout()
plt.savefig('data/solar_error_temporal_correlation.png', dpi=300)
print("Temporal-correlation heatmap saved: data/solar_error_temporal_correlation.png")
plt.show()

## 5. Fit the marginal distribution with KDE

In [ ]:
print("\n=== 5. Fit the marginal distribution with KDE ===")

# Flatten the forecast-error data to build the marginal sample set
marginal_error = forecast_error_matrix.flatten()

# Fit the KDE model
kde = gaussian_kde(marginal_error)

# Generate a smooth PDF curve
x = np.linspace(np.min(marginal_error), np.max(marginal_error), 100)
kde_pdf = kde.pdf(x)

# Fit a Gaussian distribution for comparison
mu, std = np.mean(marginal_error), np.std(marginal_error)
p = norm.pdf(x, mu, std)

In [ ]:
# Visualize the marginal distribution
plt.figure(figsize=(14, 5))
plt.hist(marginal_error, bins=100, alpha=0.5, label='Original samples', density=True)
plt.plot(x, kde_pdf, 'k', linewidth=2, label='Gaussian KDE')
plt.plot(x, p, 'r', linewidth=2, label='Gaussian fit')
plt.xlabel('Normalized forecasting error')
plt.ylabel('Density')
plt.legend()
plt.title('Solar Forecasting Error Marginal Distribution')
plt.tight_layout()
plt.savefig('data/solar_error_marginal_distribution.png', dpi=300)
print("Marginal-distribution figure saved: data/solar_error_marginal_distribution.png")
plt.show()

## 6. Generate temporally correlated samples with a Copula method

In [ ]:
print("\n=== 6. Generate temporally correlated samples with a Copula method ===")

# Set the number of generated samples
n_scenario = 200000
print(f"Number of generated scenarios: {n_scenario}")

In [ ]:
# 6.1 Generate multivariate Gaussian samples
mvnorm = multivariate_normal(mean=np.zeros(temp_corr.shape[0]), cov=temp_corr)
samples_gaussian = mvnorm.rvs(size=n_scenario, random_state=0)
print(f"Multivariate Gaussian sample shape: {samples_gaussian.shape}")

In [ ]:
# 6.2 Transform into uniform samples
norm_dist = norm()
samples_uniform = norm_dist.cdf(samples_gaussian)
print(f"Uniform sample shape: {samples_uniform.shape}")

In [ ]:
# 6.3 Construct the KDE CDF and inverse CDF with improved memory efficiency
print("Computing the KDE CDF in batches to reduce memory usage...")

stdev = np.sqrt(kde.covariance)[0, 0]
xmax = np.max(marginal_error)
xmin = np.min(marginal_error)
minmax_dist = xmax - xmin

# Generate the x-axis grid for the CDF
xx = np.linspace(xmin - 0.*minmax_dist, xmax + 0.*minmax_dist, 5000)

# Estimate the KDE CDF in batches to save memory
n_resample = 50000  # reduce the number of resampled points
n = kde.resample(n_resample, seed=0).flatten()

# Compute the CDF in batchesavoid creating a large matrix in one shot
batch_size = 1000
kde_cdf = np.zeros(len(xx))
for i in range(0, len(xx), batch_size):
    batch_end = min(i + batch_size, len(xx))
    xx_batch = xx[i:batch_end]
    # compute the CDF for the current batch
    kde_cdf[i:batch_end] = ndtr(np.subtract.outer(xx_batch, n)/stdev).mean(axis=1)
    if (i // batch_size) % 5 == 0:
        print(f"  Progress: {batch_end}/{len(xx)}")

print("KDE CDF computation completed")

In [ ]:
# Construct the inverse CDF function
kde_cdf_inv_func = interpolate.interp1d(kde_cdf, xx, kind='cubic', 
                                        bounds_error=False, fill_value='extrapolate')

In [ ]:
# 6.4 Generate the final temporally correlated KDE samples
samples_kde_corr = np.vstack([kde_cdf_inv_func(samples_uniform[:, i]) 
                               for i in range(temp_corr.shape[0])]).T
print(f"Final temporally correlated sample shape: {samples_kde_corr.shape}")

## 7. Validate the generated samples

In [ ]:
print("\n=== 7. Validate the generated samples ===")

# 7.1 Compare marginal distributions
plt.figure(figsize=(14, 5))
plt.hist(samples_kde_corr.flatten(), bins=100, alpha=0.5, 
         label='Generated samples', density=True)
plt.hist(marginal_error, bins=100, alpha=0.5, 
         label='Original samples', density=True, color='olive')
plt.plot(x, p, 'r', linewidth=2, label='Gaussian fit')
plt.xlim(np.percentile(marginal_error, 1), np.percentile(marginal_error, 99))
plt.xlabel('Normalized forecasting error')
plt.ylabel('Density')
plt.legend()
plt.title('Comparison of Generated and Original Samples')
plt.tight_layout()
plt.savefig('data/solar_error_comparison.png', dpi=300)
print("Sample-comparison figure saved: data/solar_error_comparison.png")
plt.show()

In [ ]:
# 7.2 Compare temporal correlations
temp_corr_gen = np.corrcoef(samples_kde_corr, rowvar=False)

fig, axs = plt.subplots(1, 2, figsize=(16, 7))
axs[0].set_title('Original samples')
sns.heatmap(temp_corr, annot=False, fmt=".2f", ax=axs[0],
            cmap='RdYlBu_r', vmin=-1, vmax=1, square=True,
            linewidths=0, cbar_kws={'shrink': 0.8})
axs[1].set_title('Generated samples')
sns.heatmap(temp_corr_gen, annot=False, fmt=".2f", ax=axs[1],
            cmap='RdYlBu_r', vmin=-1, vmax=1, square=True,
            linewidths=0, cbar_kws={'shrink': 0.8})
axs[0].set_xlabel('Time')
axs[0].set_ylabel('Time')
axs[1].set_xlabel('Time')
plt.tight_layout()
plt.savefig('data/solar_error_temporal_correlation_comparison.png', dpi=300)
print("Temporal-correlation comparison figure saved: data/solar_error_temporal_correlation_comparison.png")
plt.show()


In [ ]:
# 7.2 Compare temporal correlations
temp_corr_gen = np.corrcoef(samples_kde_corr, rowvar=False)

fig, axs = plt.subplots(1, 2, figsize=(16, 7))
axs[0].set_title('Original samples', fontsize=16, fontweight='bold')
sns.heatmap(temp_corr, annot=False, fmt=".2f", ax=axs[0],
            cmap='RdBu_r', vmin=-0.1, vmax=0.9, square=True,
            linewidths=0)
axs[1].set_title('Generated samples', fontsize=16, fontweight='bold')
sns.heatmap(temp_corr_gen, annot=False, fmt=".2f", ax=axs[1],
            cmap='RdBu_r', vmin=-0.1, vmax=0.9, square=True,
            linewidths=0)
axs[0].set_xlabel('Time', fontsize=14, fontweight='bold')
axs[0].set_ylabel('Time', fontsize=14, fontweight='bold')
axs[1].set_xlabel('Time', fontsize=14, fontweight='bold')
axs[1].set_ylabel('')
plt.tight_layout()

# Save both PNG (`data/`) and PDF (`paper/`) outputs for synchronized downstream use
plt.savefig('data/solar_error_temporal_correlation_comparison.png', dpi=300)
plt.savefig('paper/fig_solar_correlation_placeholder.pdf', bbox_inches='tight')
print("Temporal-correlation comparison figure saved:")
print("  data/solar_error_temporal_correlation_comparison.png")
print("  paper/fig_solar_correlation_placeholder.pdf")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from mpl_toolkits.axes_grid1 import make_axes_locatable

# --- Example data (make sure the required variables are already defined) ---
# temp_corr = np.random.rand(24, 24)
# temp_corr_gen = np.random.rand(24, 24)

# 1. Create the figure canvas
fig, axs = plt.subplots(1, 2, figsize=(16, 7))

# Define the shared heatmap configuration
heatmap_args = {
    'cmap': 'RdBu_r',
    'vmin': -0.1,
    'vmax': 0.9,
    'square': True,       # force square heatmap cells
    'linewidths': 0,
    'annot': False,
    'cbar': False         # disable the default colorbar first so it can be controlled manually
}

# --- subplots ---
sns.heatmap(temp_corr, ax=axs[0], **heatmap_args)
axs[0].set_title('Original samples', fontsize=16, fontweight='bold')
axs[0].set_xlabel('Time', fontsize=14, fontweight='bold')
axs[0].set_ylabel('Time', fontsize=14, fontweight='bold')

# Core step: use the divider to create an axis to the right of `axs[0]` with the same height for the colorbar
divider0 = make_axes_locatable(axs[0])
cax0 = divider0.append_axes("right", size="5%", pad=0.1) # `size` controls the colorbar width and `pad` controls the spacing
plt.colorbar(axs[0].get_children()[0], cax=cax0) # Draw the first heatmap colorbar on `cax0`


# --- subplots ---
sns.heatmap(temp_corr_gen, ax=axs[1], **heatmap_args)
axs[1].set_title('Generated samples', fontsize=16, fontweight='bold')
axs[1].set_xlabel('Time', fontsize=14, fontweight='bold')
axs[1].set_ylabel('')

# subplots
divider1 = make_axes_locatable(axs[1])
cax1 = divider1.append_axes("right", size="5%", pad=0.1)
plt.colorbar(axs[1].get_children()[0], cax=cax1)


# 2. Adjust the layout automatically to avoid label overlap
plt.tight_layout()

# 3. 
plt.savefig('data/solar_error_temporal_correlation_comparison.png', dpi=300)
plt.savefig('paper/fig_solar_correlation_placeholder.pdf', bbox_inches='tight')

print("Comparison figure saved; the colorbars are perfectly aligned.")
plt.show()

In [ ]:
# 7.2 Compare temporal correlations
temp_corr_gen = np.corrcoef(samples_kde_corr, rowvar=False)

fig, axs = plt.subplots(1, 2, figsize=(16, 7))
axs[0].set_title('Original samples')
sns.heatmap(temp_corr, annot=False, fmt=".2f", ax=axs[0],
            cmap='coolwarm', vmin=-1, vmax=1, square=True,  # 
            linewidths=0, cbar_kws={'shrink': 0.8})
axs[1].set_title('Generated samples')
sns.heatmap(temp_corr_gen, annot=False, fmt=".2f", ax=axs[1],
            cmap='coolwarm', vmin=-1, vmax=1, square=True,  # 
            linewidths=0, cbar_kws={'shrink': 0.8})
axs[0].set_xlabel('Time')
axs[0].set_ylabel('Time')
axs[1].set_xlabel('Time')
plt.tight_layout()
plt.savefig('data/solar_error_temporal_correlation_comparison.png', dpi=300)
print("Temporal-correlation comparison figure saved: data/solar_error_temporal_correlation_comparison.png")
plt.show()

## 8. Save results

In [ ]:
print("\n=== 8. Save results ===")

# Save the generated forecast-error samples
np.save('data/processed/solar_temporal_correlated_error_samples.npy', samples_kde_corr)
print(f"Error samples saved: data/processed/solar_temporal_correlated_error_samples.npy")
print(f"  Shape: {samples_kde_corr.shape}")

In [ ]:
# Save the PV forecast prototype (select the period with the highest generation output)
# Note: these 48 points all belong to daytime periods with generation output
solar_forecast_prototype = forecast_daytime[:48]  # use the first 48 active-generation points as the prototype

np.save('data/processed/solar_forecast_prototype.npy', solar_forecast_prototype)
print(f"PV forecast prototype saved: data/processed/solar_forecast_prototype.npy")
print(f"  Shape: {solar_forecast_prototype.shape}")
print(f": These 48 points correspond to daytime generation-period forecast values")

## Summary and usage notes

In [ ]:
print("\n=== PV error generalization completed ===")
print(f" {n_scenario} PV error scenarios")
print(f"Each scenario contains {num_time_points}  time points, all within daytime generation periods")
print("\n=== Usage notes ===")
print("When constructing a full 24-hour scenario:")
print("  1. Set nighttime periods (19:00-04:00) directly to zero")
print("  2. For daytime periods (05:00-18:00), use forecast + error sample scaled by the forecast standard deviation")
print("  3. Ensure the final scenario is nonnegative with clip(0, None)")